In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import seaborn as sns

uber_data=pd.read_csv('/kaggle/input/uberdata/UberDataset.csv')
uber_data.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/uberdata/UberDataset.csv'

# look data 

In [ ]:
uber_data.shape

In [ ]:
uber_data.info()

In [ ]:
uber_data.describe()

In [ ]:
uber_data.isnull().sum()

# data propessing
we can see that there are lots of missing values in PURPOSE column

In [ ]:
uber_data['PURPOSE'].fillna("NaN",inplace=True)

# covert to date time
Changing the START_DATE and END_DATE to the date_time format so that further it can be use to do analysis.

In [ ]:
uber_data['START_DATE']=pd.to_datetime(uber_data['START_DATE'], errors='coerce')
uber_data['END_DATE']=pd.to_datetime(uber_data['END_DATE'],errors='coerce')

Splitting the START_DATE to date and time column and then converting the time into four different categories i.e. Morning, Afternoon, Evening, Night

In [ ]:
from datetime import datetime

uber_data['Date']=pd.DatetimeIndex(uber_data['START_DATE']).date
uber_data['Time']=pd.DatetimeIndex(uber_data['START_DATE']).hour

# changeing categories
uber_data['day-time']=pd.cut(x=uber_data['Time'],
                            bins=[0,10,15,19,24],
                            labels=['Morning','afternoon','Evening','Night'])

In [ ]:
uber_data.dropna(inplace=True)

It is also important to drop the duplicates rows from the dataset. To do that, refer the code below.

In [ ]:
uber_data.drop_duplicates(inplace=True)

In [ ]:
uber_data.isnull().sum()

now, we have done eda part 
# Data visualisation

Let's start with checking the unique values in dataset of the columns with object datatype.

In [ ]:
obj = (uber_data.dtypes == 'object')
object_cols = list(obj[obj].index)

unique_values = {}
for col in object_cols:
  unique_values[col] = uber_data[col].unique().size
unique_values

 countplot the CATEGORY and PURPOSE columns.


In [ ]:
plt.figure(figsize=(10,5))

# Plot CATEGORY counts
plt.subplot(1,2,1)
sns.countplot(x='CATEGORY', data=uber_data)
plt.xticks(rotation=90)
plt.title("Trip Category Counts")

# Plot PURPOSE counts
plt.subplot(1,2,2)
sns.countplot(x='PURPOSE', data=uber_data)
plt.xticks(rotation=90)
plt.title("Trip Purpose Counts")

plt.show()




In [ ]:
# Plot day-time counts separately


plt.figure(figsize=(6,4))
sns.countplot(x='day-time', data=uber_data)
plt.title("Trips by Day Time")
plt.show()

Now, we will be comparing the two different categories along with the PURPOSE of the user.

In [ ]:
plt.figure(figsize=(15, 5))
sns.countplot(data=uber_data, x='PURPOSE', hue='CATEGORY')
plt.xticks(rotation=90)
plt.show()

# Insights from the above count-plots : 
Most of the rides are booked for business purpose.
Most of the people book cabs for Meetings and Meal / Entertain purpose.
Most of the cabs are booked in the time duration of 10am-5pm (Afternoon).

As we have seen that CATEGORY and PURPOSE columns are two very important columns. So now we will be using OneHotEncoder to categories them.

In [ ]:
# Columns to encode
object_cols = ['CATEGORY', 'PURPOSE']

# One-Hot Encoding
OH_encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
OH_cols = pd.DataFrame(OH_encoder.fit_transform(uber_data[object_cols]))

# Keep same index
OH_cols.index = uber_data.index
OH_cols.columns = OH_encoder.get_feature_names_out()

# Drop original categorical columns
df_final = uber_data.drop(object_cols, axis=1)

# ✅ Merge encoded columns with rest of the data
dataset = pd.concat([df_final, OH_cols], axis=1)

# Select numerical columns
numeric_dataset = dataset.select_dtypes(include=['number'])

After that, we can now find the correlation between the columns using heatmap.

In [ ]:


# Plot heatmap
plt.figure(figsize=(12,8))
sns.heatmap(
    numeric_dataset.corr(),
    cmap='BrBG',
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar=True
)

plt.title("Correlation Heatmap of Numerical + Encoded Features", fontsize=16, pad=20)
plt.show()


Insights from the heatmap:

Business and Personal Category are highly negatively correlated, this have already proven earlier. So this plot, justifies the above conclusions.
There is not much correlation between the features.

Now, as we need to visualize the month data. This can we same as done before (for hours). 

In [ ]:
uber_data['MONTH'] = pd.DatetimeIndex(dataset['START_DATE']).month
month_label = {1.0: 'Jan', 2.0: 'Feb', 3.0: 'Mar', 4.0: 'April',
               5.0: 'May', 6.0: 'June', 7.0: 'July', 8.0: 'Aug',
               9.0: 'Sep', 10.0: 'Oct', 11.0: 'Nov', 12.0: 'Dec'}
dataset["MONTH"] = uber_data.MONTH.map(month_label)

mon = uber_data.MONTH.value_counts(sort=False)

# Month total rides count vs Month ride max count
df = pd.DataFrame({"MONTHS": mon.values,
                   "VALUE COUNT": dataset.groupby('MONTH',
                                                  sort=False)['MILES'].max()})

p = sns.lineplot(data=df)
p.set(xlabel="MONTHS", ylabel="VALUE COUNT")

Insights from the above plot : 

The counts are very irregular.

Still its very clear that the counts are very less during Nov, Dec, Jan, which justifies the fact that  time winters are there in Florida, US.

# Visualization for days data.

In [ ]:
uber_data['Day']=uber_data.START_DATE.dt.weekday
day_label={
    0:'Mon',1:'Tue',3:'Wed',4:'Thus',5:'fri',6:'Sat',7:'Sun'
}
uber_data['Day']=uber_data['Day'].map(day_label)

day_label=uber_data.Day.value_counts()
sns.barplot(x=day_label.index, y=day_label)
plt.x_label('Day')
plt.ylabel('Count')




Now, let's explore the MILES Column .

We can use boxplot to check the distribution of the column.

In [ ]:
sns.boxplot(uber_data['MILES'])

As the graph is not clearly understandable. Let's zoom in it for values lees than 100.

In [ ]:
sns.boxplot(dataset[dataset['MILES']<100]['MILES'])

It's bit visible. But to get more clarity we can use distplot for values less than 40.

In [ ]:
sns.distplot(dataset[dataset['MILES']<40]['MILES'])

Insights from the above plots :

>Most of the cabs booked for the distance of 4-5 miles.

>Majorly people chooses cabs for the distance of 0-20 miles.

>For distance more than 20 miles cab counts is nearly negligible.
